# dots.ocr Reading-Order Inspection Notebook

`dots.ocr-1.5` produces semantic document blocks rather than line detections. This notebook evaluates its natural task: recognizing visible text blocks in the correct reading order.

Workflow:

1. Select one noisy PolyDocBench scan and its transformed GT.
2. Build visible GT semantic blocks from exported reading order.
3. Request semantic blocks from `dots.ocr`, preserving returned order.
4. Measure ordered text accuracy and block-order accuracy.
5. Inspect matched blocks and optional geometry overlays.

## 1. Environment Setup

Install optional dependencies, then create `.env` in the project root (next to `pyproject.toml`). The file is ignored by Git and is shared with the batch experiment runner:

```powershell
uv pip install -e ".[dotsocr]"
Copy-Item .env.example .env
# Open .env locally and replace sk-... with your token.
```

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Environment file: {ENV_PATH}")

## 2. Select One Scan

The input scan is taken from the prepared Tesseract experiment dataset only because that pipeline already generated the noisy images and transformed GT. No Tesseract predictions are used here.

In [ ]:
LANGUAGE_CODE = "en"
ARTICLE_ID = "history_russia"
TEMPLATE = "scientific_paper"
PAGE_NUMBER = 1
NOISE_PROFILE = "medium_scan"
VARIANT = 0

INPUT_ROOT = PROJECT_ROOT / "outputs" / "experiments" / "tesseract_quality"
RESULT_ROOT = PROJECT_ROOT / "outputs" / "experiments" / "dotsocr_ordering"
PAGE_DIR = f"page_{PAGE_NUMBER:03d}"
STEM = f"{NOISE_PROFILE}_{VARIANT}"
SOURCE_CASE = INPUT_ROOT / LANGUAGE_CODE / ARTICLE_ID / TEMPLATE / "noisy" / PAGE_DIR
RESULT_CASE = RESULT_ROOT / LANGUAGE_CODE / ARTICLE_ID / TEMPLATE / "noisy" / PAGE_DIR
SCAN_PATH = SOURCE_CASE / f"{STEM}.jpg"
GT_PATH = SOURCE_CASE / f"{STEM}_gt.json"
PREDICTION_PATH = RESULT_CASE / f"{STEM}_dotsocr_blocks.json"
RAW_RESPONSE_PATH = RESULT_CASE / f"{STEM}_dotsocr_raw.txt"
OVERLAY_PATH = RESULT_CASE / f"{STEM}_ordering_overlay.jpg"

assert SCAN_PATH.exists(), f"Missing scan: {SCAN_PATH}"
assert GT_PATH.exists(), f"Missing GT: {GT_PATH}"
print("Scan:", SCAN_PATH)
print("GT:", GT_PATH)
print("Predictions:", PREDICTION_PATH)

## 3. Build Visible GT Blocks

The GT helper groups only visible text lines of this image by their source `parent_id`, restores line order with `line_index`, and orders semantic blocks by exported block `reading_order`. This avoids evaluating text that continues on a different page.

In [ ]:
from polydocbench.eval import extract_visible_gt_blocks, load_gt

scan = Image.open(SCAN_PATH).convert("RGB")
gt = load_gt(GT_PATH)
gt_blocks = extract_visible_gt_blocks(gt, page_number=1)

print("Visible GT blocks:", len(gt_blocks))
for index, block in enumerate(gt_blocks, start=1):
    print(f"{index:02d} | {block['id']} | {block['category']} | {block['text'][:100]}")
display(IPyImage(filename=str(SCAN_PATH)))

## 4. Configure Remote Model

The model is asked for semantic blocks in reading order. Its returned array order is the prediction being evaluated; do not sort it geometrically afterward. Set `REUSE_EXISTING_PREDICTION = True` after one successful request to continue analysis without another API call.

In [ ]:
from openai import OpenAI

from polydocbench.eval.dotsocr import DEFAULT_DOTSOCR_BASE_URL, DEFAULT_DOTSOCR_MODEL, DOTSOCR_ORDERING_PROMPT

BASE_URL = DEFAULT_DOTSOCR_BASE_URL
MODEL_NAME = DEFAULT_DOTSOCR_MODEL
REQUEST_TIMEOUT = 180.0
MAX_RETRIES = 1
REUSE_EXISTING_PREDICTION = False

client = None
if not REUSE_EXISTING_PREDICTION:
    api_key = os.environ.get("LITELLM_API_KEY")
    assert api_key, "Set LITELLM_API_KEY in the project-root .env file before making a remote request"
    client = OpenAI(api_key=api_key, base_url=BASE_URL, timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES)

print("Model:", MODEL_NAME)
print("Reuse saved blocks:", REUSE_EXISTING_PREDICTION)
print("Prompt:\n", DOTSOCR_ORDERING_PROMPT)

## 5. Request Or Load Semantic Blocks

A TLS handshake timeout here indicates a gateway/network problem before model inference, not an evaluation failure.

In [ ]:
from openai import APIConnectionError, APITimeoutError
from polydocbench.eval import extract_dotsocr_blocks

if REUSE_EXISTING_PREDICTION:
    predicted_blocks = json.loads(PREDICTION_PATH.read_text(encoding="utf-8"))
else:
    try:
        predicted_blocks = extract_dotsocr_blocks(
            SCAN_PATH, client=client, model=MODEL_NAME, raw_response_path=RAW_RESPONSE_PATH
        )
    except (APITimeoutError, APIConnectionError):
        display(Markdown("**Connection failed:** check the gateway, VPN/network access, and `BASE_URL`, then retry."))
        raise
    PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
    PREDICTION_PATH.write_text(json.dumps(predicted_blocks, ensure_ascii=False, indent=2), encoding="utf-8")

print("Predicted semantic blocks:", len(predicted_blocks))
for index, block in enumerate(predicted_blocks, start=1):
    print(f"{index:02d} | {block['category']} | {block['text'][:100]}")

## 6. Evaluate Text In Reading Order

- `ordered_CER` / `ordered_WER` evaluate recognized text after block concatenation in sequence order.
- `token_F1` ignores ordering and helps separate recognition errors from sequencing errors.
- `kendall_tau` / `pairwise_accuracy` evaluate the order of text-matched semantic blocks.
- `matched_block_ratio` reports coverage and must be interpreted alongside ordering scores.

In [ ]:
from polydocbench.eval import evaluate_semantic_ordering, join_ordered_text

MIN_BLOCK_SIMILARITY = 0.30
MAX_GT_SPAN = 3
metrics, matches = evaluate_semantic_ordering(
    gt_blocks, predicted_blocks, min_similarity=MIN_BLOCK_SIMILARITY, max_gt_span=MAX_GT_SPAN
)
display(Markdown("### Ordering Metrics"))
display(metrics)

print("\n--- Ordered GT text preview ---")
print(join_ordered_text(gt_blocks)[:1000])
print("\n--- Ordered dots.ocr text preview ---")
print(join_ordered_text(predicted_blocks)[:1000])

## 7. Inspect Block Matching

Matching is text-first, with a small geometry tie-breaker. A single semantic model block may match multiple adjacent GT blocks when the model groups content more broadly.

In [ ]:
for index, match in enumerate(matches, start=1):
    print(f"--- Match {index} | similarity={match.similarity:.3f} | GT={match.gt_ids}")
    print("GT      :", " ".join(block["text"] for block in match.gt_blocks)[:300])
    print("dots.ocr:", match.prediction["text"][:300])

## 8. Diagnostic Overlay

Red rectangles show visible GT text blocks; green rectangles show semantic blocks returned by dots.ocr. This view is diagnostic only: text and returned sequence are the primary evaluation signals.

In [ ]:
def draw_bbox(draw, bbox, color: str, width: int = 3):
    x, y = float(bbox["x"]), float(bbox["y"])
    draw.rectangle([x, y, x + float(bbox["width"]), y + float(bbox["height"])], outline=color, width=width)

overlay = scan.copy()
draw = ImageDraw.Draw(overlay)
for block in gt_blocks:
    draw_bbox(draw, block["bbox"], "red")
for block in predicted_blocks:
    draw_bbox(draw, block["bbox"], "green")
OVERLAY_PATH.parent.mkdir(parents=True, exist_ok=True)
overlay.save(OVERLAY_PATH)
print("Overlay:", OVERLAY_PATH)
display(IPyImage(filename=str(OVERLAY_PATH)))

if RAW_RESPONSE_PATH.exists():
    print("\nRaw response preview:\n", RAW_RESPONSE_PATH.read_text(encoding="utf-8")[:3000])